# Today Day 8, I am doing community Detection, without Neo4j's Paid graph Data Science Tier
## A Real Contraint Worth confirming before writing any code    

The obvious way to do this, Neo4j's `CALL gds.louvain.stream(...)`, running Louvain community detection natively inside the database, doesn't work on the tier I've actually been using. Confirmed directly from Neo4j's own community forum: "*at the moment, there is no free offering for GDS in Aura.*" The Graph Data Science library requires a separate, paid AuraDS instance for a self-hosted Neo4j with the GDS plugin intslled. Writing today's code around `gds.*` procedures would produce something that fails the moment I run it. The real fix: Pulling the graph's structure out through an ordinary **Cypher query**, Build the equivalent graph in **Python** with `networkx`, and run the same underlying algorithm there instead. Same technique, executed client-side rather than inside the database.

# Coding Time! Export, Detect, Write Back 

**Build a Co- Occurence Signal From What's Actually in the Graph:** There's no existing edge directly connecting two `Entity` nodes for "these come up together", `POSSIBLY_SAME_AS` (Day 2) only covers ambiguous near-duplicates, a different, much narrower thing. Today derives a real signal from the schema that already exists: tro entities that the *same user* has asked about, weighted by how many users share that pattern.

**Export to `networkx`, Run Real Louvain:** Pull that co-occurence data through Cypher, build a weighted `networkx` graph client-side, and run `network`'ss own Louvain implementation, the same algorithm GDS would have run, just not inside the database.

**Write Detected Communities Back:** Store Each entity's `community_id` as a property on the real Neo4j node, so any future Cypher query can filter or group by community directly, without re-running detection every time.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [2]:
# Connecting to a real Graph database

from neo4j import GraphDatabase

# AuraDB free tier: get these three values from your Neo4j Aura console
# after creating a free instance — no local install needed
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with neo4j_driver.session() as session:
    result = session.run("RETURN 'connected' AS status")
    print(f"✅ Neo4j: {result.single()['status']}")

✅ Neo4j: connected


In [3]:
# Verification Gate before building anything new

with neo4j_driver.session() as session:
    total_entities = session.run("MATCH (e:Entity) RETURN count(e) AS c").single()["c"]
    total_users = session.run("MATCH (u:User) RETURN count(u) AS c").single()["c"]

print(f"Entities in graph: {total_entities} | Users: {total_users}")
assert total_entities > 0, (
    "❌ No entities exist yet. If Day 6's Fix 2A/2B never actually landed, "
    "nothing has been written to this graph — go confirm Day 6 before continuing here."
)
print("✅ Graph has real data — proceeding")

Entities in graph: 3 | Users: 1
✅ Graph has real data — proceeding


In [4]:
# Exporting the Co-Occurence Graph

import networkx as nx

with neo4j_driver.session() as session:
    pairs = list(session.run(
        """
        MATCH (e1:Entity)<-[:ASKED_ABOUT]-(u:User)-[:ASKED_ABOUT]->(e2:Entity)
        WHERE id(e1) < id(e2)
        RETURN e1.name AS entity1, e2.name AS entity2, count(DISTINCT u) AS weight
        """
    ))
    all_entities = list(session.run("MATCH (e:Entity) RETURN e.name AS name"))

G = nx.Graph()
for row in all_entities:
    G.add_node(row["name"])
for row in pairs:
    G.add_edge(row["entity1"], row["entity2"], weight=row["weight"])

print(f"📋 Exported graph: {G.number_of_nodes()} entities, {G.number_of_edges()} co-occurrence edges")
if G.number_of_edges() == 0:
    print("⚠️  No edges found — community detection needs at least some entities shared across")
    print("   users or sessions. With only one test user so far, this is expected; real signal")
    print("   will appear once multiple real sessions/users have real overlapping questions.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=3, column=15, offset=94>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 94, 'line': 3, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (e1:Entity)<-[:ASKED_ABOUT]-(u:User)-[:ASKED_ABOUT]->(e2:Entity)\n        WHERE id(e1) < id(e2)\n        RETURN e1.name AS entity1, e2.name AS entity2, count(DISTINCT u) AS weight\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feat

📋 Exported graph: 3 entities, 0 co-occurrence edges
⚠️  No edges found — community detection needs at least some entities shared across
   users or sessions. With only one test user so far, this is expected; real signal
   will appear once multiple real sessions/users have real overlapping questions.


In [5]:
# Run Real Louvain Community Detection

import community as community_louvain

if G.number_of_edges() > 0:
    partition = community_louvain.best_partition(G, weight="weight")
    num_communities = len(set(partition.values()))
    print(f"✅ Detected {num_communities} communities across {len(partition)} entities")

    from collections import defaultdict
    by_community = defaultdict(list)
    for entity, community_id in partition.items():
        by_community[community_id].append(entity)

    for cid, entities in by_community.items():
        print(f"\nCommunity {cid}:")
        for e in entities:
            print(f"  • {e}")
else:
    partition = {name: 0 for name in G.nodes()}
    print("⚠️  No edges — everything defaulted to a single community")

⚠️  No edges — everything defaulted to a single community


In [6]:
# Writing Communities Back to The Real Graph

with neo4j_driver.session() as session:
    for entity_name, community_id in partition.items():
        session.run(
            "MATCH (e:Entity {name: $name}) SET e.community_id = $cid",
            name=entity_name, cid=community_id
        )

print(f"✅ Wrote community_id to {len(partition)} real Entity nodes")
print("   Any future Cypher query can now do: MATCH (e:Entity {community_id: X}) ...")
print("   without re-running detection")

✅ Wrote community_id to 3 real Entity nodes
   Any future Cypher query can now do: MATCH (e:Entity {community_id: X}) ...
   without re-running detection


In [7]:
# Verifying Gainst Real Data by reading a Community Back

with neo4j_driver.session() as session:
    check = list(session.run(
        "MATCH (e:Entity) WHERE e.community_id IS NOT NULL "
        "RETURN e.community_id AS cid, collect(e.name) AS entities ORDER BY cid"
    ))

for row in check:
    print(f"Community {row['cid']}: {row['entities']}")

Community 0: ['direct_customer_segments', 'Services segment revenue FY2024', 'capital return program']
